### ベイズ線形回帰の定式化

scikit-learnに対応する回帰クラスが存在しないので詳しく記載します。



#### 関数定義

n次元多変量正規分布（Multivariate normal distribution）定義
$$
P(x|\mu,\Sigma) = \frac{1}{C} \exp( -\frac{1}{2} (x-\mu)^T \Sigma^{-1} (x-\mu) )
$$
$\mu$は平均値。$\Sigma$ は分散共分散行列。
ここに規格化定数は 
$$ C=\sqrt{(2\pi)^n|\Sigma|}$$
である。

#### y(x,w)とガウスノイズ

目的変数$y(x,w)$は基底関数$x$と係数$w$の線形方程式で書けるとする。
$$
y(x) = x^T w
$$
観測値にはyに更にノイズが入るとし、
y(x,w)とガウスノイズから目的変数$t$が得られているとする。
$$
t = N(y(x,w),\beta)
$$

観測値がある値ベクトルtを取る確率は
$t$,$x$の各成分$t_i$, $x_i$を用いて
$$
P(t|x,w)
= \exp \left( -\frac{1}{2} \sum_i (t_i - x_i^T w)^T \beta_i^{-1} (t_i - x_i^T w ) \right) \mbox{,(1-1)}
$$
ともかける。今は$\beta_i=\beta$である。
以下
では$x$,$\beta$を顕に書くのをやめてこれを $P(t|w)$書いておく。

#### ベイズの定理

今、tが与えられた時にwを求めたい。
$P(t|w)$から$P(w|t)$への変換はベイズの定理を用いて
$$
P(w|t) = P(t|w) P(w) /P(t)
$$
と書ける。
$P(t)$ は$P(w|t)$を計算する上では定数なので$P(t)=1$としておく。
つまり
$$
P(w|t) \propto P(t|w) P(w)
$$
である。
$P(w)$の部分をprior, $P(t|w)$の部分をlikelihood,$P(w|t)$をposteriorと呼ぶ。

##### 直接解法
wに関する確率を知りたいので$P(t|x,w )$のexpの中をwで平方完成する。

まずwの二次は
$$
w_T (\sum_i x_i \beta_i^{-1} x_i^T ) w \mbox{, (1-2)}
$$

wの一次は
$$
w_T \sum_i x_i \beta_i^{-1} t_i + h.c. \mbox{, (1-3)}
$$
となる。
式(2)と式(3)の$w$の一次と二次の項で書けるということは，
$\sum_i$の和を取った結果，何かの$\bar S_N^{-1}$と$\bar m_N$を用いて
$$
P(w|t)_N = \exp \left( -\frac{1}{2} (w -  \bar m_N )^T \bar S_N^{-1} (w - \bar m_N ) \right) \mbox{,(1-4)}
$$
とも書けるということと等価である．
式(12)と式(1-3)で$\bar S_N^{-1}$と$\bar m_N$を評価するのを直接解法とする。

##### 逐次解法
異なる解法として式(1-1)の和をexpの外に出して以下に書く．
$$
P(t|x,w)
= \Pi_i \exp \left( -\frac{1}{2} (t_i - x_i^T w)^T \beta_i^{-1} (t_i - x_i^T w ) \right) \mbox{,(2-1)}
$$
$i$はデータインスタンスのindexであり，新たにデータインスタンスが
観測されると$P(t|x,w)$が変化していくとみなせる．
更に，
$$
S_i^{-1} = x_i \beta_i^{-1} x_i^T
$$
$$
m_i =  S_i x_i \beta_i^{-1} t_i
$$
と定義すると式(1)は
$$
P(t|x,w) = \exp ( -\frac{1}{2} \sum_i 
(w -  m_i )^T  S_i^{-1} (w-   m_i ) )
$$
である。expを分割して
$$
P(t|x,w) = \Pi_i \exp \left( -\frac{1}{2} 
(w -    m_i )^T  S_i^{-1} (w-  m_i ) \right)
$$
とした方がわかりやすいかもしれない。ここで$S_i^{-1}$は対称行列であることを用いて式変形をしている。
$(x_i \beta_i^{-1} x_i^T)$は(wのサイズ) x (wのサイズ)の行列である。



priorがｗに関して多変量正規分布に従う。つまり、
$$
P(w) = N(w|m,S_0)
 = \exp \left( -\frac{1}{2} ( w- m_0)^T S_0^{-1} (w-m_0) \right) 
$$
の形で書けると**仮定**する。

$i$=0をpriorとし、観測値が$1-N$ 個あるとする．
$N$番目ということを顕に書き、N個の観測値を使ったposterior（式(2-1))を
$$
P(w|t )_N = \Pi_{i=0}^{N} \exp \left( -\frac{1}{2} 
(w -   m_i )^T  S_i^{-1} (w-  m_i ) \right)
$$
と書き直す．$P(w|t )_N$は漸化式の形
$$
P(w|t )_N =  \exp \left( -\frac{1}{2} 
(w -    m_N )^T  S_N^{-1} (w-   m_N ) \right) P(w|t )_{N-1} \mbox{,(2-2)}
$$
に書けることは理解できると思う．


式(1-4)と式(2-2)は同じ式の異なる表現であり，expの中$w$の二次と一次の項を比べると$\bar{S}_N$と$\bar{m}_N$の以下の漸近式が得られる。
$$
\bar S_N^{-1} = S_N^{-1} + \bar S_{N-1}^{-1} = x_N \beta_N^{-1} x_N^T + \bar S_{N-1}^{-1} 
$$
$$
\bar m_N =  \bar S_N ( S_{N}^{-1} m_{N} + \bar S_{N-1}^{-1} \bar m_{N-1} ) =  \bar S_N ( x_{N}\beta_N^{-1} t_{N} + \bar S_{N-1}^{-1} \bar m_{N-1} ) 　
$$

（$\bar{S}_N^{-1}$の逆行列演算が必要なことに注意．）